# Parameter Golf — TT+MLA+GQA Training\n\n**Single-file Colab notebook.**  No uploads, no Drive, no GitHub required.\n\nSub-16MB language model with structural compression baked at initialization:\n- **TT** (Tensor Train): 8× parameter reduction on Q/O projections\n- **MLA** (Multi-head Latent Attention): 32× KV cache reduction\n- **GQA** (Grouped-Query Attention): 8:1 query-to-KV head ratio\n\n| Resource | Requirement |\n|----------|-------------|\n| GPU | Colab Pro A100 40GB |\n| Training time | ~2-4 hours |\n| Final artifact | <2 MB compressed (<16 MB limit) |\n| Parameters | 4,370,704 (~8.74 MB BF16) |

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers tokenizers datasets tqdm numpy\n\nimport torch\nprint(f\"PyTorch {torch.__version__}\")\nprint(f\"CUDA: {torch.cuda.is_available()}\")\nif torch.cuda.is_available():\n    print(f\"GPU: {torch.cuda.get_device_name(0)}\")\n    print(f\"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB\")

## 2. Write All Source Files

In [ ]:
import os, pathlib\n\nFILES = {}\n\n# ── tt_layers.py ──\nFILES['tt_layers.py'] = r'''\n"""\nTensor Train (TT) decomposed linear layer.\nReplaces a dense (d, d) weight matrix with a chain of small TT cores,\nachieving ~8x parameter reduction.\n"""\nimport math\nimport torch\nimport torch.nn as nn\n\nclass TTLinear(nn.Module):\n    def __init__(self, in_features, out_features, tt_rank=16, bias=True):\n        super().__init__()\n        assert in_features == out_features, \"TTLinear requires square weight matrix\"\n        self.in_features = in_features\n        self.out_features = out_features\n        self.tt_rank = tt_rank\n        self.factor = int(math.isqrt(in_features))\n        if self.factor * self.factor != in_features:\n            raise ValueError(f\"in_features ({in_features}) must be a perfect square\")\n        self.core1 = nn.Parameter(torch.empty(self.factor, self.factor, tt_rank))\n        self.core2 = nn.Parameter(torch.empty(tt_rank, self.factor, self.factor))\n        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None\n        if bias:\n            self.bias = nn.Parameter(torch.empty(out_features))\n        else:\n            self.register_parameter('bias', None)\n        self.reset_parameters()\n\n    def reset_parameters(self):\n        std = 1.0 / math.sqrt(self.tt_rank * self.factor)\n        nn.init.normal_(self.core1, std=std)\n        nn.init.normal_(self.core2, std=std)\n        if self.bias is not None:\n            nn.init.zeros_(self.bias)\n\n    def forward(self, x):\n        *batch, _ = x.shape\n        x = x.reshape(-1, self.factor, self.factor)\n        h = torch.einsum(\"b i j, k i r -> b k j r\", x, self.core1)\n        y = torch.einsum(\"b k j r, r l j -> b k l\", h, self.core2)\n        y = y.reshape(*batch, self.out_features)\n        if self.bias is not None:\n            y = y + self.bias\n        return y\n'''\n\n# ── mla_attention.py ──\nFILES['mla_attention.py'] = r'''\n"""Multi-head Latent Attention with GQA and RoPE."""\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass MLAAttention(nn.Module):\n    def __init__(self, d_model=256, n_heads=8, n_kv_heads=1, d_head=32, d_c=16, max_seq=512):\n        super().__init__()\n        self.d_model = d_model\n        self.n_heads = n_heads\n        self.n_kv_heads = n_kv_heads\n        self.d_head = d_head\n        self.d_c = d_c\n        self.heads_per_kv = n_heads // n_kv_heads\n        self.max_seq = max_seq\n        self.W_kv_down = nn.Linear(d_model, d_c, bias=False)\n        self.W_k_up = nn.Linear(d_c, n_kv_heads * d_head, bias=False)\n        self.W_v_up = nn.Linear(d_c, n_kv_heads * d_head, bias=False)\n        self._build_rope_cache()\n        self._init_weights()\n\n    def _build_rope_cache(self):\n        theta = 10000.0\n        dim = self.d_head\n        freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))\n        t = torch.arange(self.max_seq).float()\n        freqs = torch.outer(t, freqs)\n        self.register_buffer(\"rope_cos\", freqs.cos())\n        self.register_buffer(\"rope_sin\", freqs.sin())\n\n    def _init_weights(self):\n        nn.init.normal_(self.W_kv_down.weight, std=1.0 / math.sqrt(self.d_model))\n        nn.init.normal_(self.W_k_up.weight, std=1.0 / math.sqrt(self.d_c))\n        nn.init.normal_(self.W_v_up.weight, std=1.0 / math.sqrt(self.d_c))\n\n    def _apply_rope(self, x):\n        B, H, T, D = x.shape\n        half = D // 2\n        x_r = x.reshape(B, H, T, half, 2)\n        x_even, x_odd = x_r[..., 0], x_r[..., 1]\n        cos = self.rope_cos[:T].view(1, 1, T, half)\n        sin = self.rope_sin[:T].view(1, 1, T, half)\n        rot_even = x_even * cos - x_odd * sin\n        rot_odd = x_odd * cos + x_even * sin\n        return torch.stack([rot_even, rot_odd], dim=-1).reshape(B, H, T, D)\n\n    def forward(self, x, q):\n        B, T, _ = x.shape\n        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)\n        q = self._apply_rope(q)\n        kv_latent = self.W_kv_down(x)\n        k = self.W_k_up(kv_latent).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)\n        k = self._apply_rope(k)\n        v = self.W_v_up(kv_latent).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)\n        if self.n_kv_heads < self.n_heads:\n            k = k.expand(-1, self.heads_per_kv, -1, -1).reshape(B, self.n_heads, T, self.d_head)\n            v = v.expand(-1, self.heads_per_kv, -1, -1).reshape(B, self.n_heads, T, self.d_head)\n        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True, scale=1.0/math.sqrt(self.d_head))\n        return attn_out.transpose(1, 2).contiguous().view(B, T, -1)\n'''\n\n# ── model.py ──\nFILES['model.py'] = r'''\n"""TT+MLA+GQA Transformer model — ~4.37M params."""\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom tt_layers import TTLinear\nfrom mla_attention import MLAAttention\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim, eps=1e-6):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(dim))\n    def forward(self, x):\n        dtype = x.dtype\n        x = x.float()\n        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)\n        return (x / rms).to(dtype) * self.weight\n\nclass SwiGLUFFN(nn.Module):\n    def __init__(self, d_model=256, ffn_dim=512):\n        super().__init__()\n        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)\n        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)\n        self._init_weights()\n    def _init_weights(self):\n        std_h = 1.0 / math.sqrt(self.gate_proj.in_features)\n        std_o = 1.0 / math.sqrt(self.down_proj.in_features)\n        nn.init.normal_(self.gate_proj.weight, std=std_h)\n        nn.init.normal_(self.up_proj.weight, std=std_h)\n        nn.init.normal_(self.down_proj.weight, std=std_o)\n    def forward(self, x):\n        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))\n\nclass TransformerBlock(nn.Module):\n    def __init__(self, d_model=256, n_heads=8, n_kv_heads=1, d_head=32, d_c=16, ffn_dim=512, tt_rank=16, max_seq=512):\n        super().__init__()\n        self.norm1 = RMSNorm(d_model)\n        self.W_q = TTLinear(d_model, d_model, tt_rank=tt_rank, bias=False)\n        self.attention = MLAAttention(d_model, n_heads, n_kv_heads, d_head, d_c, max_seq)\n        self.W_o = TTLinear(d_model, d_model, tt_rank=tt_rank, bias=False)\n        self.attn_scale = nn.Parameter(torch.ones(1))\n        self.norm2 = RMSNorm(d_model)\n        self.ffn = SwiGLUFFN(d_model, ffn_dim)\n        self.mlp_scale = nn.Parameter(torch.ones(1))\n    def forward(self, x):\n        normed = self.norm1(x)\n        q = self.W_q(normed)\n        attn_out = self.attention(normed, q)\n        attn_out = self.W_o(attn_out)\n        x = x + attn_out * self.attn_scale\n        normed = self.norm2(x)\n        x = x + self.ffn(normed) * self.mlp_scale\n        return x\n\nclass TTMLATransformer(nn.Module):\n    def __init__(self, vocab_size=4096, d_model=256, n_layers=8, n_heads=8, n_kv_heads=1, d_head=32, d_c=16, ffn_dim=512, tt_rank=16, max_seq=512):\n        super().__init__()\n        self.vocab_size = vocab_size\n        self.d_model = d_model\n        self.n_layers = n_layers\n        self.max_seq = max_seq\n        self.embedding = nn.Embedding(vocab_size, d_model)\n        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, n_kv_heads, d_head, d_c, ffn_dim, tt_rank, max_seq) for _ in range(n_layers)])\n        self.norm_final = RMSNorm(d_model)\n        nn.init.normal_(self.embedding.weight, std=1.0)\n    def forward(self, input_ids, return_loss=False, targets=None):\n        x = self.embedding(input_ids)\n        for block in self.blocks:\n            x = block(x)\n        x = self.norm_final(x)\n        logits = F.linear(x, self.embedding.weight)\n        if return_loss:\n            return F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))\n        return logits\n    def count_parameters(self):\n        total = sum(p.numel() for p in self.parameters())\n        return total, total\n'''\n\n# ── tokenizer_.py ──\nFILES['tokenizer_.py'] = r'''\n"""BPE tokenizer training + byte-counting LUT builder."""\nimport math, json, os\nfrom pathlib import Path\nimport torch\n\ndef train_bpe_tokenizer(output_path, vocab_size=4096, sample_size=100_000_000):\n    from tokenizers import Tokenizer, models, trainers, pre_tokenizers\n    from datasets import load_dataset\n    print(f\"Training BPE tokenizer (vocab={vocab_size})...\")\n    print(\"  Downloading FineWeb-Edu sample...\")\n    ds = load_dataset(\"HuggingFaceFW/fineweb-edu\", \"sample-10BT\", streaming=True, split=\"train\")\n    samples, chars = [], 0\n    for row in ds:\n        text = row[\"text\"]\n        samples.append(text)\n        chars += len(text)\n        if chars >= sample_size:\n            break\n    print(f\"  Collected {len(samples):,} documents ({chars:,} chars)\")\n    tokenizer = Tokenizer(models.BPE(unk_token=\"<unk>\"))\n    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)\n    trainer_obj = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=[\"<unk>\", \"<s>\", \"</s>\", \"<pad>\"], min_frequency=2, show_progress=True)\n    tokenizer.train_from_iterator(samples, trainer_obj)\n    tokenizer.save(str(output_path))\n    print(f\"  Tokenizer saved, vocab size: {tokenizer.get_vocab_size()}\")\n    return tokenizer\n\ndef build_byte_luts(tokenizer, vocab_size=4096):\n    if isinstance(tokenizer, (str, Path)):\n        from tokenizers import Tokenizer as T\n        tokenizer = T.from_file(str(tokenizer))\n    vocab = tokenizer.get_vocab()\n    id_to_token = {idx: tok for tok, idx in vocab.items()}\n    special_ids = set(vocab.get(t, -1) for t in [\"<unk>\", \"<s>\", \"</s>\", \"<pad>\"])\n    base_bytes_lut = torch.zeros(vocab_size, dtype=torch.int16)\n    has_leading_space = torch.zeros(vocab_size, dtype=torch.bool)\n    is_boundary = torch.zeros(vocab_size, dtype=torch.bool)\n    for idx in range(vocab_size):\n        token_str = id_to_token.get(idx)\n        if token_str is None or idx in special_ids:\n            is_boundary[idx] = True\n            continue\n        decoded = token_str\n        if decoded.startswith(\"\\u0120\"):\n            has_leading_space[idx] = True\n            decoded = decoded[1:]\n        base_bytes_lut[idx] = len(decoded.encode(\"utf-8\"))\n    return {\"base_bytes_lut\": base_bytes_lut, \"has_leading_space\": has_leading_space, \"is_boundary\": is_boundary}\n\ndef compute_val_bpb(val_loss, val_token_count, val_byte_count):\n    bits_per_token = val_loss / math.log(2.0)\n    tokens_per_byte = val_token_count / max(val_byte_count, 1)\n    return bits_per_token * tokens_per_byte\n'''\n\n# ── data_pipeline.py ──\nFILES['data_pipeline.py'] = r'''\n"""FineWeb-Edu streaming data pipeline."""\nimport torch\nfrom torch.utils.data import IterableDataset, DataLoader\n\nclass FineWebDataset(IterableDataset):\n    def __init__(self, tokenizer, seq_len=512, split=\"train\", max_docs=None):\n        super().__init__()\n        self.tokenizer = tokenizer\n        self.seq_len = seq_len\n        self.split = split\n        self.max_docs = max_docs\n        self.bos_id = tokenizer.token_to_id(\"<s>\") or 0\n        self.eos_id = tokenizer.token_to_id(\"</s>\") or 2\n    def __iter__(self):\n        from datasets import load_dataset\n        ds = load_dataset(\"HuggingFaceFW/fineweb-edu\", \"sample-10BT\", streaming=True, split=self.split)\n        buffer, doc_count = [], 0\n        for row in ds:\n            text = row[\"text\"]\n            if not text or len(text.strip()) < 10:\n                continue\n            tokens = self.tokenizer.encode(text).ids\n            if len(tokens) < 4:\n                continue\n            buffer.append(self.bos_id)\n            buffer.extend(tokens)\n            buffer.append(self.eos_id)\n            doc_count += 1\n            if self.max_docs and doc_count >= self.max_docs:\n                break\n            while len(buffer) >= self.seq_len + 1:\n                chunk = buffer[:self.seq_len + 1]\n                buffer = buffer[self.seq_len:]\n                yield torch.tensor(chunk[:-1], dtype=torch.long), torch.tensor(chunk[1:], dtype=torch.long)\n        while len(buffer) >= self.seq_len + 1:\n            chunk = buffer[:self.seq_len + 1]\n            buffer = buffer[self.seq_len:]\n            yield torch.tensor(chunk[:-1], dtype=torch.long), torch.tensor(chunk[1:], dtype=torch.long)\n\ndef create_dataloader(tokenizer, seq_len=512, batch_size=64, split=\"train\", num_workers=2, prefetch_factor=4, max_docs=None):\n    dataset = FineWebDataset(tokenizer, seq_len, split, max_docs)\n    def collate_fn(batch):\n        return torch.stack([b[0] for b in batch]), torch.stack([b[1] for b in batch])\n    return DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers, prefetch_factor=prefetch_factor if num_workers > 0 else None, pin_memory=True)\n'''\n\n# ── eval.py ──\nFILES['eval.py'] = r'''\n"""Validation evaluation — bits-per-byte (BPB) computation."""\nimport math\nimport torch\nimport torch.nn.functional as F\n\ndef compute_val_bpb(val_loss, val_token_count, val_byte_count):\n    val_loss_mean = val_loss / max(val_token_count, 1)\n    bits_per_token = val_loss_mean / math.log(2.0)\n    tokens_per_byte = val_token_count / max(val_byte_count, 1)\n    return val_loss_mean, bits_per_token * tokens_per_byte\n\ndef count_token_bytes(target_ids, prev_ids, luts):\n    base = luts[\"base_bytes_lut\"][target_ids].long()\n    leading = luts[\"has_leading_space\"][target_ids]\n    boundary_prev = luts[\"is_boundary\"][prev_ids]\n    extra = leading & ~boundary_prev\n    return (base + extra.long()).sum().item()\n\n@torch.no_grad()\ndef evaluate_model(model, token_ids, luts, seq_len, device):\n    model.eval()\n    total_tokens = token_ids.shape[0]\n    usable = (total_tokens - 1) // seq_len * seq_len\n    token_ids = token_ids[:usable + 1]\n    val_loss_sum = 0.0\n    val_token_count = 0\n    val_byte_count = 0\n    with torch.amp.autocast(\"cuda\", dtype=torch.bfloat16):\n        for i in range(0, usable, seq_len):\n            chunk = token_ids[i:i + seq_len + 1]\n            inputs = chunk[:-1].unsqueeze(0).to(device)\n            targets = chunk[1:].unsqueeze(0).to(device)\n            prev_ids = chunk[:seq_len].to(device)\n            logits = model(inputs)\n            loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1), reduction=\"sum\")\n            val_loss_sum += loss.item()\n            val_token_count += seq_len\n            tgt_cpu = targets.view(-1).cpu()\n            prev_cpu = prev_ids.view(-1).cpu()\n            val_byte_count += count_token_bytes(tgt_cpu, prev_cpu, luts)\n    return compute_val_bpb(val_loss_sum, val_token_count, val_byte_count)\n\n@torch.no_grad()\ndef evaluate_sliding_window(model, token_ids, luts, seq_len, stride, device):\n    model.eval()\n    total_tokens = token_ids.shape[0]\n    val_loss_sum = 0.0\n    val_token_count = 0\n    val_byte_count = 0\n    with torch.amp.autocast(\"cuda\", dtype=torch.bfloat16):\n        for start in range(0, total_tokens - seq_len, stride):\n            end = start + seq_len\n            chunk = token_ids[start:end + 1]\n            inputs = chunk[:-1].unsqueeze(0).to(device)\n            targets = chunk[1:].unsqueeze(0).to(device)\n            count_start = 0 if start == 0 else seq_len - stride\n            scored_targets = targets[:, count_start:]\n            scored_prev = chunk[count_start:seq_len].to(device)\n            logits = model(inputs)\n            scored_logits = logits[:, count_start:, :]\n            loss = F.cross_entropy(scored_logits.reshape(-1, scored_logits.shape[-1]), scored_targets.reshape(-1), reduction=\"sum\")\n            n_scored = scored_targets.numel()\n            val_loss_sum += loss.item()\n            val_token_count += n_scored\n            tgt_cpu = scored_targets.view(-1).cpu()\n            prev_cpu = scored_prev.view(-1).cpu()\n            val_byte_count += count_token_bytes(tgt_cpu, prev_cpu, luts)\n    return compute_val_bpb(val_loss_sum, val_token_count, val_byte_count)\n\ndef load_validation_tokens(tokenizer, num_tokens=1_000_000):\n    from datasets import load_dataset\n    print(f\"  Loading validation data (~{num_tokens:,} tokens)...\")\n    ds = load_dataset(\"HuggingFaceFW/fineweb-edu\", \"sample-10BT\", streaming=True, split=\"train\")\n    all_tokens = []\n    for row in ds:\n        text = row[\"text\"]\n        if not text or len(text.strip()) < 10:\n            continue\n        encoded = tokenizer.encode(text)\n        all_tokens.extend(encoded.ids)\n        if len(all_tokens) >= num_tokens:\n            break\n    tokens = torch.tensor(all_tokens[:num_tokens], dtype=torch.long)\n    print(f\"  Loaded {tokens.shape[0]:,} validation tokens\")\n    return tokens\n'''\n\n# ── quantize.py ──\nFILES['quantize.py'] = r'''\n"""int8 per-row quantization + zlib level 9 compression."""\nimport io, math, zlib\nfrom pathlib import Path\nimport torch\n\nCONTROL_PATTERNS = (\"attn_scale\", \"attn_scales\", \"mlp_scale\", \"mlp_scales\", \"resid_mix\", \"resid_mixes\", \"q_gain\", \"skip_weight\", \"skip_weights\")\nSMALL_TENSOR_THRESHOLD = 65536\nQFORMAT = \"int8_clean_per_row_v1\"\n\ndef _is_control(name):\n    return any(p in name for p in CONTROL_PATTERNS)\n\ndef quantize_float_tensor(t):\n    if t.dim() == 2:\n        t_abs = t.float().abs()\n        k = max(1, int(t.shape[1] * 99.99984 / 100.0))\n        clip_abs = t_abs.kthvalue(t.shape[1] - k + 1, dim=1).values.clamp(min=1.0/127.0)\n        scales = (clip_abs / 127.0).to(torch.float16)\n        t_clipped = t.float().clamp(-clip_abs.unsqueeze(1), clip_abs.unsqueeze(1))\n        q = torch.round(t_clipped / scales.unsqueeze(1)).clamp(-127, 127).to(torch.int8)\n        return q, scales, True\n    else:\n        t_abs = t.float().abs()\n        clip_abs = t_abs.max().clamp(min=1.0/127.0)\n        scale = (clip_abs / 127.0).to(torch.float16)\n        q = torch.round(t.float().clamp(-clip_abs, clip_abs) / scale).clamp(-127, 127).to(torch.int8)\n        return q, torch.tensor([scale], dtype=torch.float16), False\n\ndef quantize_state_dict(state_dict):\n    quantized, scales, dtypes, passthrough, passthrough_orig_dtypes, qmeta = {}, {}, {}, {}, {}, {}\n    for name, param in state_dict.items():\n        if param.dtype not in (torch.float32, torch.float16, torch.bfloat16):\n            passthrough[name] = param.clone()\n            passthrough_orig_dtypes[name] = str(param.dtype)\n            continue\n        if param.numel() <= SMALL_TENSOR_THRESHOLD or _is_control(name):\n            passthrough[name] = param.to(torch.float16).clone()\n            passthrough_orig_dtypes[name] = str(param.dtype)\n            continue\n        q, s, is_per_row = quantize_float_tensor(param)\n        quantized[name] = q\n        scales[name] = s\n        dtypes[name] = \"int8\"\n        qmeta[name] = {\"per_row\": is_per_row}\n    return {\"__quant_format__\": QFORMAT, \"quantized\": quantized, \"scales\": scales, \"dtypes\": dtypes, \"passthrough\": passthrough, \"passthrough_orig_dtypes\": passthrough_orig_dtypes, \"qmeta\": qmeta}\n\ndef dequantize_state_dict(quant_dict):\n    result = {}\n    for name, q in quant_dict.get(\"quantized\", {}).items():\n        s = quant_dict[\"scales\"][name]\n        if quant_dict[\"qmeta\"][name][\"per_row\"]:\n            s = s.view(-1, *([1]*(q.dim()-1)))\n        result[name] = q.float() * s.float()\n    for name, t in quant_dict.get(\"passthrough\", {}).items():\n        result[name] = t.clone()\n    return result\n\ndef compress_state_dict(quant_dict):\n    buf = io.BytesIO()\n    torch.save(quant_dict, buf)\n    return zlib.compress(buf.getvalue(), level=9)\n\ndef decompress_state_dict(blob):\n    return torch.load(io.BytesIO(zlib.decompress(blob)), map_location=\"cpu\", weights_only=False)\n\ndef save_compressed_model(model, output_path, code_path=None):\n    print(\"Quantizing model...\")\n    quant_dict = quantize_state_dict(model.state_dict())\n    qp = sum(quant_dict[\"quantized\"][k].numel() for k in quant_dict[\"quantized\"])\n    pp = sum(quant_dict[\"passthrough\"][k].numel() for k in quant_dict[\"passthrough\"])\n    print(f\"  Quantized params:   {qp:,}\")\n    print(f\"  Passthrough params: {pp:,}\")\n    blob = compress_state_dict(quant_dict)\n    Path(output_path).write_bytes(blob)\n    quant_file_bytes = len(blob)\n    print(f\"  Compressed size:    {quant_file_bytes:,} bytes ({quant_file_bytes/1e6:.2f} MB)\")\n    raw_size = sum(p.numel() * p.element_size() for p in model.parameters())\n    compression_ratio = raw_size / max(quant_file_bytes, 1)\n    code_bytes = 0\n    if code_path:\n        code_bytes = len(Path(code_path).read_text(\"utf-8\").encode(\"utf-8\"))\n        print(f\"  Code size:          {code_bytes:,} bytes ({code_bytes/1e6:.2f} MB)\")\n    bytes_total = quant_file_bytes + code_bytes\n    print(f\"  Total artifact:     {bytes_total:,} bytes ({bytes_total/1e6:.2f} MB)\")\n    limit = 16_000_000\n    print(f\"  Status:             {'UNDER 16MB' if bytes_total < limit else f'OVER by {bytes_total - limit:,} bytes'}\")\n    return {\"quant_file_bytes\": quant_file_bytes, \"code_bytes\": code_bytes, \"bytes_total\": bytes_total, \"compression_ratio\": compression_ratio}\n\ndef load_and_dequantize(ptz_path):\n    blob = Path(ptz_path).read_bytes()\n    return dequantize_state_dict(decompress_state_dict(blob))\n'''\n\n# ── train_gpt.py ──\nFILES['train_gpt.py'] = r'''\n"""Parameter Golf — TT+MLA+GQA Training Script."""\nimport math, os, sys, time\nfrom pathlib import Path\nimport torch\nimport torch.nn.functional as F\nfrom model import TTMLATransformer\nfrom tokenizer_ import train_bpe_tokenizer, build_byte_luts\nfrom data_pipeline import create_dataloader\nfrom eval import evaluate_model, evaluate_sliding_window, load_validation_tokens\nfrom quantize import save_compressed_model, load_and_dequantize\n\ndef get_config():\n    return {\n        \"vocab_size\": 4096, \"d_model\": 256, \"n_layers\": 8, \"n_heads\": 8, \"n_kv_heads\": 1,\n        \"d_head\": 32, \"d_c\": 16, \"ffn_dim\": 512, \"tt_rank\": 16, \"max_seq\": 512,\n        \"batch_size\": int(os.getenv(\"BATCH_SIZE\", \"64\")),\n        \"seq_len\": int(os.getenv(\"SEQ_LEN\", \"512\")),\n        \"lr\": float(os.getenv(\"LR\", \"6e-3\")),\n        \"weight_decay\": 0.1, \"beta1\": 0.9, \"beta2\": 0.95, \"grad_clip\": 1.0,\n        \"warmup_steps\": int(os.getenv(\"WARMUP_STEPS\", \"500\")),\n        \"train_steps\": int(os.getenv(\"TRAIN_STEPS\", \"10000\")),\n        \"decay_steps\": int(os.getenv(\"DECAY_STEPS\", \"1000\")),\n        \"eval_every\": int(os.getenv(\"EVAL_EVERY\", \"1000\")),\n        \"checkpoint_every\": int(os.getenv(\"CHECKPOINT_EVERY\", \"2000\")),\n        \"max_val_tokens\": int(os.getenv(\"MAX_VAL_TOKENS\", \"200000\")),\n        \"sliding_window_stride\": 64,\n        \"compile_model\": bool(int(os.getenv(\"COMPILE_MODEL\", \"1\"))),\n        \"dtype\": torch.bfloat16,\n        \"output_dir\": Path(os.getenv(\"OUTPUT_DIR\", \"./output\")),\n        \"tokenizer_path\": Path(\"./tokenizer.json\"),\n    }\n\ndef get_lr(step, cfg):\n    warmup, total, decay, peak = cfg[\"warmup_steps\"], cfg[\"train_steps\"], cfg[\"decay_steps\"], cfg[\"lr\"]\n    min_lr = peak * 0.1\n    if step < warmup: return peak * step / max(warmup, 1)\n    elif step < total - decay: return peak\n    else:\n        frac = (step - (total - decay)) / max(decay, 1)\n        return peak - (peak - min_lr) * frac\n\n@torch.no_grad()\ndef run_eval(model, tokenizer, luts, device, cfg, sliding=False):\n    val_tokens = load_validation_tokens(tokenizer, cfg[\"max_val_tokens\"])\n    if sliding:\n        return evaluate_sliding_window(model, val_tokens, luts, cfg[\"seq_len\"], cfg[\"sliding_window_stride\"], device)\n    return evaluate_model(model, val_tokens, luts, cfg[\"seq_len\"], device)\n\ndef train(cfg):\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"Device: {device}\")\n    if device.type == \"cuda\":\n        print(f\"GPU: {torch.cuda.get_device_name(0)}\")\n        print(f\"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB\")\n    output_dir = cfg[\"output_dir\"]\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    print(\"\\n=== Step 1: Tokenizer ===\")\n    tokenizer_path = cfg[\"tokenizer_path\"]\n    if tokenizer_path.exists():\n        from tokenizers import Tokenizer\n        tokenizer = Tokenizer.from_file(str(tokenizer_path))\n        print(f\"Loaded tokenizer from {tokenizer_path}\")\n    else:\n        tokenizer = train_bpe_tokenizer(output_path=tokenizer_path, vocab_size=cfg[\"vocab_size\"])\n    actual_vocab = tokenizer.get_vocab_size()\n    luts = build_byte_luts(tokenizer, vocab_size=max(cfg[\"vocab_size\"], actual_vocab))\n    print(f\"Vocab size: {actual_vocab}, Byte LUTs built\")\n\n    print(\"\\n=== Step 2: Model ===\")\n    model = TTMLATransformer(vocab_size=cfg[\"vocab_size\"], d_model=cfg[\"d_model\"], n_layers=cfg[\"n_layers\"], n_heads=cfg[\"n_heads\"], n_kv_heads=cfg[\"n_kv_heads\"], d_head=cfg[\"d_head\"], d_c=cfg[\"d_c\"], ffn_dim=cfg[\"ffn_dim\"], tt_rank=cfg[\"tt_rank\"], max_seq=cfg[\"max_seq\"])\n    model = model.to(device)\n    total, _ = model.count_parameters()\n    print(f\"Parameters: {total:,} ({total*2/1e6:.2f} MB BF16)\")\n    if cfg[\"compile_model\"] and device.type == \"cuda\":\n        print(\"Compiling model...\")\n        model = torch.compile(model, mode=\"reduce-overhead\")\n\n    print(\"\\n=== Step 3: Optimizer ===\")\n    decay_p, nodecay_p = [], []\n    for name, p in model.named_parameters():\n        if not p.requires_grad: continue\n        (nodecay_p if (p.dim() < 2 or \"norm\" in name or \"scale\" in name or \"embedding\" in name) else decay_p).append(p)\n    optimizer = torch.optim.AdamW([{\"params\": decay_p, \"weight_decay\": cfg[\"weight_decay\"]}, {\"params\": nodecay_p, \"weight_decay\": 0.0}], lr=cfg[\"lr\"], betas=(cfg[\"beta1\"], cfg[\"beta2\"]))\n    print(f\"AdamW, lr={cfg['lr']}, wd={cfg['weight_decay']}\")\n\n    print(\"\\n=== Step 4: Data ===\")\n    train_loader = create_dataloader(tokenizer=tokenizer, seq_len=cfg[\"seq_len\"], batch_size=cfg[\"batch_size\"], num_workers=2, prefetch_factor=4)\n    print(f\"Batch: {cfg['batch_size']}, Tokens/step: {cfg['batch_size']*cfg['seq_len']:,}\")\n\n    print(f\"\\n=== Step 5: Training ({cfg['train_steps']} steps) ===\\n\")\n    model.train()\n    step, tokens_seen, best_val_bpb, t_start = 0, 0, float('inf'), time.time()\n    train_loss_accum, train_steps_accum = 0.0, 0\n    train_iter = iter(train_loader)\n    while step < cfg[\"train_steps\"]:\n        for pg in optimizer.param_groups: pg[\"lr\"] = get_lr(step, cfg)\n        try:\n            input_ids, targets = next(train_iter)\n        except StopIteration:\n            train_iter = iter(train_loader)\n            input_ids, targets = next(train_iter)\n        input_ids, targets = input_ids.to(device), targets.to(device)\n        with torch.amp.autocast(\"cuda\", dtype=cfg[\"dtype\"]):\n            loss = model(input_ids, return_loss=True, targets=targets)\n        optimizer.zero_grad()\n        loss.backward()\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg[\"grad_clip\"])\n        optimizer.step()\n        train_loss_accum += loss.item()\n        train_steps_accum += 1\n        tokens_seen += input_ids.numel()\n        if step % 100 == 0 and step > 0:\n            elapsed = time.time() - t_start\n            print(f\"  step {step:>6d}/{cfg['train_steps']} | loss {train_loss_accum/train_steps_accum:.4f} | lr {get_lr(step, cfg):.2e} | grad {grad_norm:.2f} | {tokens_seen/max(elapsed,0.001)/1e6:.1f}M tok/s | {elapsed:.0f}s\")\n            train_loss_accum, train_steps_accum = 0.0, 0\n        if step > 0 and step % cfg[\"eval_every\"] == 0:\n            val_loss, val_bpb = run_eval(model, tokenizer, luts, device, cfg)\n            print(f\"  >>> EVAL step {step}: val_loss={val_loss:.4f}  val_bpb={val_bpb:.4f} <<<\")\n            if val_bpb < best_val_bpb:\n                best_val_bpb = val_bpb\n                torch.save(model.state_dict(), output_dir / \"best_model.pt\")\n                print(f\"  >>> New best val_bpb: {val_bpb:.4f} (saved) <<<\")\n            model.train()\n        if step > 0 and step % cfg[\"checkpoint_every\"] == 0:\n            torch.save({\"step\": step, \"model_state_dict\": model.state_dict(), \"best_val_bpb\": best_val_bpb, \"tokens_seen\": tokens_seen}, output_dir / f\"ckpt_{step}.pt\")\n        step += 1\n\n    print(f\"\\n=== Step 6: Final eval ===\")\n    total_time = time.time() - t_start\n    print(f\"Time: {total_time:.0f}s ({total_time/60:.1f}m), Tokens: {tokens_seen:,}\")\n    best_ckpt = output_dir / \"best_model.pt\"\n    if best_ckpt.exists():\n        model.load_state_dict(torch.load(best_ckpt, map_location=device))\n    val_loss, val_bpb = run_eval(model, tokenizer, luts, device, cfg)\n    sw_loss, sw_bpb = run_eval(model, tokenizer, luts, device, cfg, sliding=True)\n    print(f\"Final val_bpb: {val_bpb:.4f}  |  Sliding-window: {sw_bpb:.4f}\")\n\n    print(f\"\\n=== Step 7: Quantize ===\")\n    ptz_path = output_dir / \"final_model.int8.ptz\"\n    info = save_compressed_model(model, ptz_path, code_path=Path(__file__))\n    print(\"\\nRoundtrip validation...\")\n    sd = load_and_dequantize(ptz_path)\n    model.load_state_dict(sd, strict=True)\n    rt_loss, rt_bpb = run_eval(model, tokenizer, luts, device, cfg)\n    print(f\"Roundtrip val_bpb: {rt_bpb:.4f}\")\n    print(f\"Roundtrip exact:   {rt_loss:.8f}  {rt_bpb:.8f}\")\n    limit = 16_000_000\n    print(f\"\\n{'='*60}\")\n    print(f\"Training complete.\")\n    print(f\"  Best val_bpb:      {best_val_bpb:.4f}\")\n    print(f\"  Roundtrip val_bpb: {rt_bpb:.4f}\")\n    print(f\"  Artifact:          {info['bytes_total']:,} bytes ({info['bytes_total']/1e6:.2f} MB)\")\n    print(f\"  Under 16MB:        {info['bytes_total'] < limit}\")\n    print(f\"  Time:              {total_time:.0f}s\")\n    print(f\"  Tokens seen:       {tokens_seen:,}\")\n    print(f\"{'='*60}\")\n'''\n\n# ── Write all files ──\nfor filename, content in FILES.items():\n    path = pathlib.Path(filename)\n    path.write_text(content.lstrip('\\n'))\n    print(f\"  Wrote {filename} ({len(content)} chars)\")\n\nprint(\"\\nAll source files written.\")

## 3. Quick Architecture Check

In [ ]:
from model import TTMLATransformer\nimport torch\n\nmodel = TTMLATransformer()\ntotal, _ = model.count_parameters()\nprint(f\"Parameters: {total:,}  ({total*2/1e6:.2f} MB BF16)\")\nprint(f\"Under 16MB: {total * 2 < 16_000_000}\")\n\ndevice = torch.device('cuda')\nmodel = model.to(device)\nx = torch.randint(0, 4096, (4, 512), device=device)\ny = torch.randint(0, 4096, (4, 512), device=device)\nwith torch.no_grad():\n    loss = model(x, return_loss=True, targets=y)\nprint(f\"Initial loss: {loss.item():.2f} (random ~ln(4096)=8.32)\")\ndel model, x, y; torch.cuda.empty_cache()\nprint(\"\\nModel OK!\")

## 4. Train Tokenizer

In [ ]:
from tokenizer_ import train_bpe_tokenizer, build_byte_luts\n\ntokenizer = train_bpe_tokenizer(\n    output_path='./tokenizer.json',\n    vocab_size=4096,\n    sample_size=100_000_000,\n)\nluts = build_byte_luts(tokenizer, vocab_size=4096)\nprint(f\"Vocab: {tokenizer.get_vocab_size()}, LUTs ready\")

## 5. Train Model

In [ ]:
import os\n# --- TUNE THESE ---\nos.environ['BATCH_SIZE'] = '128'\nos.environ['TRAIN_STEPS'] = '10000'    # ~3.3B tokens\nos.environ['LR'] = '6e-3'\nos.environ['WARMUP_STEPS'] = '500'\nos.environ['DECAY_STEPS'] = '1000'\nos.environ['EVAL_EVERY'] = '1000'\nos.environ['CHECKPOINT_EVERY'] = '2000'\nos.environ['MAX_VAL_TOKENS'] = '200000'\nos.environ['OUTPUT_DIR'] = './output'\n\nprint(f\"Batch: {os.environ['BATCH_SIZE']} | Steps: {os.environ['TRAIN_STEPS']} | LR: {os.environ['LR']}\")\nprint(f\"Tokens/step: {int(os.environ['BATCH_SIZE']) * 512:,}\")\nprint(f\"Total tokens: {int(os.environ['BATCH_SIZE']) * 512 * int(os.environ['TRAIN_STEPS']):,}\")\n\nfrom train_gpt import train, get_config\ncfg = get_config()\ntrain(cfg)

## 6. Results

In [ ]:
import os, glob\noutput_dir = os.environ.get('OUTPUT_DIR', './output')\nprint(\"Output files:\")\nfor f in sorted(os.listdir(output_dir)):\n    path = os.path.join(output_dir, f)\n    size_mb = os.path.getsize(path) / 1e6\n    print(f\"  {f:40s}  {size_mb:.2f} MB\")\n\nptz_path = os.path.join(output_dir, 'final_model.int8.ptz')\nif os.path.exists(ptz_path):\n    ptz_size = os.path.getsize(ptz_path)\n    code_path = './train_gpt.py'\n    code_size = os.path.getsize(code_path) if os.path.exists(code_path) else 0\n    total = ptz_size + code_size\n    print(f\"\\n  Compressed model: {ptz_size:,} bytes ({ptz_size/1e6:.2f} MB)\")\n    print(f\"  Code:             {code_size:,} bytes ({code_size/1e6:.2f} MB)\")\n    print(f\"  Total artifact:   {total:,} bytes ({total/1e6:.2f} MB)\")\n    print(f\"  Under 16MB:       {total < 16_000_000}\")